### IDF + Word2Vec
- Word2Vec은 단문에서 효과적인 벡터화 
- TF-IDF에서 TF 하나의 문장에서 단어의 출현 횟수의 값인데 단문에서 단어들의 출현 횟수는 일반적으로 1회 정도이기 때문에 큰 의미를 가질수 없다. 
- IDF와 Word2Vec을 혼합하여 활용

In [ ]:
import numpy as np 
import pandas as pd
from gensim.models import Word2Vec
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from konlpy.tag import Komoran

In [ ]:
docs = [
    '오늘 날씨가 좋다 여행 가고 싶다', 
    '기온이 너무 올라서 아무것도 하기 싫다', 
    '수업이 너무 지루하고 졸리다', 
    '음식이 너무 맛이 없고 서비스도 별로다',
    '영화가 너무 재미있어서 시간이 가는 줄 몰랐다'
]
target = [1, 0, 0, 0, 1]

In [ ]:
# 토큰화 함수 
def tokenize(text):
    # konlpy 설치하고 토큰화 객체 생성 시 JDK필요(최신 버전에서 문제가 발생)
    # Komoran이 사용가능한 경우와 불가능한 경우 
    try:
        komoran = Komoran()
        allow_pos = ['NNP', 'NNG', 'VV', 'VA', 'SL', 'MAG']
        tokens = []
        for word, pos in komoran.pos(text):
            if pos in allow_pos:
                tokens.append(word)
        
    except Exception as e: 
        print('Komoran 사용이 불가 :', e)
        tokens = text.split()
    
    return tokens
        
        

In [ ]:
tokenize(docs[0])

In [ ]:
tokens = []
for doc in docs:
    token = tokenize(doc)
    tokens.append(token)
tokens

In [ ]:
# Word2Vec 객체 생성 
w2v = Word2Vec(
    sentences= tokens, 
    vector_size=100, 
    window = 5, 
    min_count=1, 
    epochs=100, 
    sg = 1, 
    workers=2, 
    seed=42
)

In [ ]:
wv = w2v.wv

In [ ]:
# 문장을 입력값으로 단위 벡터의 평균을 구하는 함수
def sent_embed_mean(token):
    # token : 토큰화 된 하나의 문장 
    vector = []
    for word in token:
        if word in wv.index_to_key:
            # Word2Vec에서 학습이 된 단어 사전에 word가 존재한다면
            # vector 리스트에 해당 단어의 단위 벡터를 추가 
            vector.append( wv[word] )
    # 만약에 새로운 문장의 단어들이 Word2Vec에서 사전에 학습된 단어 사전에 존재하지 않는 경우(vector의 값이 빈 리스트)
    if vector:
        # vector가 빈 리스트가 아닌 경우 
        result = np.mean(vector, axis=0)
    else:
        # vector가 존재하지 않는 경우에는 0행렬을 생성(사이즈는 wv의 vecrot_size 만큼)
        result = np.zeros(wv.vector_size)
    
    return result


In [ ]:
# tokens 데이터를 이용하여 단위 벡터 평균 함수를 호출 
X_embed_wv = []
for token in tokens:
    X_embed_wv.append(
        sent_embed_mean(token)
    )

X_embed_wv

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# 데이터를 train, test로 분할하고 생성된 모델을 매개변수로 받아서 
# 해당 모델의 학습을 하고 예측 후 평가 지표를 출력하는 함수

def run_model( X, y, model ,test_size = 0.2, stratify = None ):
    # X : 독립 변수
    # y : 종속 변수
    # model : 사용할 모델
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size= test_size, stratify= stratify, random_state=42
    )
    # 인자로 받은 모델을 이용해서 학습 
    model.fit(X_train, y_train)
    # 학습된 모델을 이용하여 예측 값을 생성 
    pred = model.predict(X_test)

    # 평가 지표를 생성 (분류 레포트)
    result = classification_report(pred, y_test)

    return result

     

In [ ]:
svc = SVC(random_state=42)

In [ ]:
print(run_model(X_embed_wv, target, svc))

In [ ]:
# Word2Vec에 idf의 값들을 포함 
# 문맥 상에서 단어의 예측 벡터와 전체 문서에서 특정 단어들의 중요도를 결합 

# TF-IDF 벡터화 행렬 생성 
tfidf_vec = TfidfVectorizer(
    tokenizer=tokenize, 
    lowercase= False
).fit(docs)

In [ ]:
# word2vec에서는 단어의 이름을 가지고 단위 벡터 생성 
# idf 값들도 단어에 따라서 추출하기 편하게 만들기 위해서 dict 형태로 데이터를 생성
idf_weight = dict(zip(
    tfidf_vec.get_feature_names_out(), 
    tfidf_vec.idf_
))
idf_weight.keys()

In [ ]:
# 단어 별 단위 벡터의 평균 값과 idf의 값들을 곱하여 새로운 벡터를 생성 
def sent_embed_wv_idf(token):
    # 단어별 단위벡터
    vector = []
    # idf 값
    idf = []

    for word in token:
        if word in wv.index_to_key and word in idf_weight:
            # print(word)
            vector.append(wv[word] * idf_weight[word])
            idf.append(idf_weight[word])
            # 각 단어별 단위 벡터에 idf를 곱한 값 -> vector의 합산과 idf의 합산을 나눠준다.(평균을 구하는 방식)
    # print(vector)
    # print(idf)
    if vector :
        # 1e-9 사용하는 이유는? -> 분모를 0으로 만들지 않기 위함(굉장히 작은 수를 더해준다. )
        result = np.sum(vector, axis=0) / (np.sum(idf) + 1e-9)
    else:
        result = np.zeros(wv.vector_size)
    
    return result

In [ ]:
X_emb_wv_idf = []
for token in tokens :
    # print(token)
    X_emb_wv_idf.append(
        sent_embed_wv_idf(token)
    )

X_emb_wv_idf

In [ ]:
print(run_model(X_emb_wv_idf, target, svc))

In [ ]:
# 학습 된 모델에서 예측 값들을 되돌려 받는 함수를 생성 
# 임베딩 기법을 선택 : w2v만을 이용한 평균 , w2v + idf 혼합한 벡터 평균

# 매개변수 3개 : 새로운 데이터( 문장들 ), 학습된 모델, 벡터화 타입 
def predict_sentence_list(
        sentences, model, vec_type = 'wv'
):
    # sentences : 문장들의 리스트 
    # model : 학습이 된 모델
    # vec_type : 'wv'나 아니면 'idf' 값을 받는다. 
    # 문장들을 토큰화 -> tokenize 함수를 호출하여 결과를 받아온다. 
    X_test = [] 
    for sentence in sentences:
        token = tokenize(sentence)
        # 하나의 문장이 토큰화가 진행 되었으면 벡터화 함수에 데이터를 대입 
        if vec_type == 'wv':
            vec = sent_embed_mean(token)
        elif vec_type == 'idf':
            vec = sent_embed_wv_idf(token)
        else:
            print('vec_type이 맞지 않습니다')
            return ''
        X_test.append(vec)      # 독립 변수 생성 완료
    
    preds = model.predict(X_test)

    result = []
    for sentence, pred in zip(sentences, preds):
        label = "긍정" if pred == 1 else '부정'
        result.append([sentence, label])
    return result
    

In [ ]:
# run_model() 함수를 이용해서 모델에 학습하고 성능 평가 
# predict_sentence_list() 함수를 이용해서 새로운 문장들을 예측하는 함수 
new_sentences = [
    '영화가 너무 지루해서 돈이 아깝다', 
    '날씨가 너무 별로다', 
    '기온이 좋아서 어디론가 떠나고 싶다'
]

In [ ]:
predict_sentence_list(new_sentences, svc, 'wv')

In [ ]:
predict_sentence_list(new_sentences, svc, 'idf')

In [66]:
# 함수 안에서 전역 변수의 데이터를 변경 
a = "Hello"

In [1]:
def func_1(text):
    # 전역 변수 a에 접근 하는 방식 1번 : globals() 함수를 이용
    # globals()['a'] = text
    # global 키워드를 이용 방식 2번
    # a라는 전역 변수를 가지고 와서 사용
    global a
    a = text
    return a

In [2]:
print(a)


NameError: name 'a' is not defined

In [3]:
print(func_1('Hi'))

Hi


In [4]:
print(a)

Hi
